#Phase 3 KISC Course Report Summary Table Creation

> This program creates a summary table that is used in Phase 3 KISC course reports. The summary table will display in the final cell of this notebook. No tables in the Databricks catalog are altered by this script. This script is purely for generating calculations for the Phase 3 KISC reports.

> The data comes from a series of KISC surveys in Qualtrics that is managed by the PECQI unit. A new KISC survey is created every block. These surveys are aggregated into a single table in Qualtrics.
>   - sandbox.pecqi.p3_kisc_tbl_prod

> Specify the parameters in the widgets above. See the sandbox.pecqi.p3_course_code_map_prod table for KISC names.

> When specifying the _Start Academic Year_ and _Start Block_, you should use the more recent academic year if the block overlaps another cohort. For example, AY 23-24 Block 16 and AY 24-25 Block 4 occur at the same time, therefore, you would specify the most recent academic year and block combination, which in this case is AY 24-25 Block 4. Likewise, use the more recent ay-block combination when specifying _End Academic Year_ and _End Block_. By specifying the Start Academic Year and Start Block in this manner, the program appropriately includes and excludes overlapping blocks. Review the output from the ay_block list below to see the combinations of ay blocks that were included in the analysis after running the script.

##Access Parameter Values

In [0]:
# Access the parameters from the widgets
kisc = dbutils.widgets.get("kisc")
end_ay = dbutils.widgets.get("end_ay")
end_block = dbutils.widgets.get("end_block")
start_ay = dbutils.widgets.get("start_ay")
start_block = dbutils.widgets.get("start_block")

##Generate Dataframe Containing AY-Block Pairs to Evaluate


In [0]:
from pyspark.sql.types import StructType, StructField, StringType

# Define the function to generate the academic year-block combinations to be analyzed
def generate_ay_blocks(start_ay, end_ay, start_block, end_block):
    # Convert block numbers to integers
    start_block_num = int(start_block.split(" ")[1])
    end_block_num = int(end_block.split(" ")[1])
    print(f"start_block_num: {start_block_num}, end_block_num: {end_block_num}") 

    # Extract academic years as integers (assuming format like "23-24")
    start_year1, start_year2 = map(int, start_ay.split("-"))
    end_year1, end_year2 = map(int, end_ay.split("-"))
    print(f"start_year1: {start_year1}, start_year2: {start_year2}, end_year1: {end_year1}, end_year2: {end_year2}")

    ay_blocks = []

    # Include overlapping blocks from the previous academic year
    if start_block_num <= 4:
        previous_start_year1 = start_year1 - 1
        previous_start_year2 = start_year2 - 1
        print(f"Previous academic year: {previous_start_year1}-{previous_start_year2}")

        for block in range(13 + (start_block_num-1), 17):
            block_entry = (f"Block {block}", f"{previous_start_year1:02d}-{previous_start_year2:02d}")
            ay_blocks.append(block_entry)
            print(f"Added from previous year: {block_entry}")

    # Include blocks within the range of start and end academic years
    current_year1 = start_year1
    current_year2 = start_year2

    for year1 in range(start_year1, end_year1 + 1):
        for block in range(1, 17):
            if (year1 == start_year1 and block < start_block_num) or (year1 == end_year1 and block > end_block_num):
                continue

            block_entry = (f"Block {block}", f"{current_year1:02d}-{current_year2:02d}")
            ay_blocks.append(block_entry)
            print(f"Added: {block_entry}")

        current_year1 += 1
        current_year2 += 1

    # Remove blocks that should not be included due to overlapping blocks of the previous academic year
    final_year_records_to_exclude = set()
    if end_block_num <= 4:
        previous_end_year1 = end_year1 - 1
        previous_end_year2 = end_year2 - 1
        for block in range(17 - (4 - end_block_num), 17):
            block_entry = (f"Block {block}", f"{previous_end_year1:02d}-{previous_end_year2:02d}")
            final_year_records_to_exclude.add(block_entry)
            print(f"Exclusion added: {block_entry}")

    ay_blocks = [x for x in ay_blocks if x not in final_year_records_to_exclude]

    return ay_blocks

# Generate the data
data = generate_ay_blocks(start_ay, end_ay, start_block, end_block)

# Define schema for the DataFrame
schema = StructType([
    StructField("block", StringType(), True),
    StructField("academic_year", StringType(), True)
])

# Create the DataFrame
ay_blocks_df = spark.createDataFrame(data, schema)

# If needed, collect the data to the driver for further use (e.g., in pandas)
ay_blocks = ay_blocks_df.collect()

In [0]:
# view ay-blocks pairs to be included in the summary table
display(ay_blocks)

##Import Data

In [0]:
# Read the kisc table into a DataFrame
df_kisc = spark.table("sandbox.pecqi.p3_kisc_tbl_prod")

##Recode and Transform Data

In [0]:
# Create a list of column names to be recoded
kisc_col_labels = [
    "overall_satisf", "director_effective", "instruct_methods", 
    "science_amount", "science_quality", "apply_learning", 
    "balance_class_clinic", "health_equity"
]

# Create a list of new column names to hold the numeric values
kisc_recode_labels = [
    "overall_satisf_num", "director_effective_num", "instruct_methods_num", 
    "science_amount_num", "science_quality_num", "apply_learning_num", 
    "balance_class_clinic_num", "health_equity_num"
]

In [0]:
from pyspark.sql.functions import col, when

# Create new numeric columns for each of the specified columns based on satisfaction level
for col_label, recode_label in zip(kisc_col_labels, kisc_recode_labels):
  df_kisc = df_kisc.withColumn(
    recode_label,
    when(col(col_label) == "Very Satisfied", 5)
    .when(col(col_label) == "Satisfied", 4)
    .when(col(col_label) == "Neutral", 3)
    .when(col(col_label) == "Dissatisfied", 2)
    .when(col(col_label) == "Very Dissatisfied", 1)
    .otherwise(None)  # Use None for unmatched entries to represent missing values
)

####Filter KISC DataFrame

In [0]:
# Filter to the specified AYs and Blocks by doing an inner join on block and academic_year
filtered_df = df_kisc.join(ay_blocks_df, on=["block", "academic_year"], how="inner")

In [0]:
# Filter out visiting student data by removing rows where visiting is 'Yes'
filtered_df = filtered_df.filter(filtered_df.visiting != 'Yes')

In [0]:
# View the filtered DataFrame prepared for analysis
display(filtered_df)

##Calculations and Aggregations

####All KISCs

In [0]:
from pyspark.sql.functions import avg

# Calculate the average values for all KISCs
avg_overall_satisf_all = round(filtered_df.agg(avg("overall_satisf_num")).collect()[0][0], 2)
avg_director_effective_all = round(filtered_df.agg(avg("director_effective_num")).collect()[0][0], 2)
avg_instruct_methods_all = round(filtered_df.agg(avg("instruct_methods_num")).collect()[0][0], 2)
avg_science_amount_all = round(filtered_df.agg(avg("science_amount_num")).collect()[0][0], 2)
avg_science_quality_all = round(filtered_df.agg(avg("science_quality_num")).collect()[0][0], 2)
avg_apply_learning_all = round(filtered_df.agg(avg("apply_learning_num")).collect()[0][0], 2)
avg_balance_class_clinic_all = round(filtered_df.agg(avg("balance_class_clinic_num")).collect()[0][0], 2)
avg_health_equity_all = round(filtered_df.agg(avg("health_equity_num")).collect()[0][0], 2)

In [0]:
from pyspark.sql.functions import count

# Calculate the count of responses for all KISCs
count_overall_satisf_all = filtered_df.agg(count("overall_satisf_num")).collect()[0][0]
count_director_effective_all = filtered_df.agg(count("director_effective_num")).collect()[0][0]
count_instruct_methods_all = filtered_df.agg(count("instruct_methods_num")).collect()[0][0]
count_science_amount_all = filtered_df.agg(count("science_amount_num")).collect()[0][0]
count_science_quality_all = filtered_df.agg(count("science_quality_num")).collect()[0][0]
count_apply_learning_all = filtered_df.agg(count("apply_learning_num")).collect()[0][0]
count_balance_class_clinic_all = filtered_df.agg(count("balance_class_clinic_num")).collect()[0][0]
count_health_equity_all = filtered_df.agg(count("health_equity_num")).collect()[0][0]

In [0]:
# Further filter the DataFrame to include only rows where KISC matches the specified value
filtered_to_kisc = filtered_df.filter(filtered_df.kisc == kisc)

# Calculate the average values for the specified KISC
avg_overall_satisf_indiv_kisc = round(filtered_to_kisc.agg(avg("overall_satisf_num")).collect()[0][0], 2)
avg_director_effective_indiv_kisc = round(filtered_to_kisc.agg(avg("director_effective_num")).collect()[0][0], 2)
avg_instruct_methods_indiv_kisc = round(filtered_to_kisc.agg(avg("instruct_methods_num")).collect()[0][0], 2)
avg_science_amount_indiv_kisc = round(filtered_to_kisc.agg(avg("science_amount_num")).collect()[0][0], 2)
avg_science_quality_indiv_kisc = round(filtered_to_kisc.agg(avg("science_quality_num")).collect()[0][0], 2)
avg_apply_learning_indiv_kisc = round(filtered_to_kisc.agg(avg("apply_learning_num")).collect()[0][0], 2)
avg_balance_class_clinic_indiv_kisc = round(filtered_to_kisc.agg(avg("balance_class_clinic_num")).collect()[0][0], 2)
avg_health_equity_indiv_kisc = round(filtered_to_kisc.agg(avg("health_equity_num")).collect()[0][0], 2)

In [0]:
# Calculate the count of responses for individual KISCs
count_overall_satisf_indiv_kisc = filtered_to_kisc.agg(count("overall_satisf_num")).collect()[0][0]
count_director_effective_indiv_kisc = filtered_to_kisc.agg(count("director_effective_num")).collect()[0][0]
count_instruct_methods_indiv_kisc = filtered_to_kisc.agg(count("instruct_methods_num")).collect()[0][0]
count_science_amount_indiv_kisc = filtered_to_kisc.agg(count("science_amount_num")).collect()[0][0]
count_science_quality_indiv_kisc = filtered_to_kisc.agg(count("science_quality_num")).collect()[0][0]
count_apply_learning_indiv_kisc = filtered_to_kisc.agg(count("apply_learning_num")).collect()[0][0]
count_balance_class_clinic_indiv_kisc = filtered_to_kisc.agg(count("balance_class_clinic_num")).collect()[0][0]
count_health_equity_indiv_kisc = filtered_to_kisc.agg(count("health_equity_num")).collect()[0][0]

In [0]:
from pyspark.sql.functions import count, avg

# Define the function to display averages and counts in a clean way
def display_averages_counts(
    avg_overall_satisf_all, avg_director_effective_all, avg_instruct_methods_all, avg_science_amount_all,
    avg_science_quality_all, avg_apply_learning_all, avg_balance_class_clinic_all, avg_health_equity_all,
    count_overall_satisf_all, count_director_effective_all, count_instruct_methods_all, count_science_amount_all,
    count_science_quality_all, count_apply_learning_all, count_balance_class_clinic_all, count_health_equity_all,
    avg_overall_satisf_indiv_kisc, avg_director_effective_indiv_kisc, avg_instruct_methods_indiv_kisc, avg_science_amount_indiv_kisc,
    avg_science_quality_indiv_kisc, avg_apply_learning_indiv_kisc, avg_balance_class_clinic_indiv_kisc, avg_health_equity_indiv_kisc,
    count_overall_satisf_indiv_kisc, count_director_effective_indiv_kisc, count_instruct_methods_indiv_kisc, count_science_amount_indiv_kisc,
    count_science_quality_indiv_kisc, count_apply_learning_indiv_kisc, count_balance_class_clinic_indiv_kisc, count_health_equity_indiv_kisc, kisc_value
):
    print(f"Averages for All Data:")
    print(f"  Average Overall Satisfaction: {avg_overall_satisf_all} (count: {count_overall_satisf_all})")
    print(f"  Average Director Effectiveness: {avg_director_effective_all} (count: {count_director_effective_all})")
    print(f"  Average Instruction Methods: {avg_instruct_methods_all} (count: {count_instruct_methods_all})")
    print(f"  Average Science Amount: {avg_science_amount_all} (count: {count_science_amount_all})")
    print(f"  Average Science Quality: {avg_science_quality_all} (count: {count_science_quality_all})")
    print(f"  Average Apply Learning: {avg_apply_learning_all} (count: {count_apply_learning_all})")
    print(f"  Average Balance Class-Clinic: {avg_balance_class_clinic_all} (count: {count_balance_class_clinic_all})")
    print(f"  Average Health Equity: {avg_health_equity_all} (count: {count_health_equity_all})")
    print()
    print(f"Averages for KISC '{kisc_value}':")
    print(f"  Average Overall Satisfaction: {avg_overall_satisf_indiv_kisc} (count: {count_overall_satisf_indiv_kisc})")
    print(f"  Average Director Effectiveness: {avg_director_effective_indiv_kisc} (count: {count_director_effective_indiv_kisc})")
    print(f"  Average Instruction Methods: {avg_instruct_methods_indiv_kisc} (count: {count_instruct_methods_indiv_kisc})")
    print(f"  Average Science Amount: {avg_science_amount_indiv_kisc} (count: {count_science_amount_indiv_kisc})")
    print(f"  Average Science Quality: {avg_science_quality_indiv_kisc} (count: {count_science_quality_indiv_kisc})")
    print(f"  Average Apply Learning: {avg_apply_learning_indiv_kisc} (count: {count_apply_learning_indiv_kisc})")
    print(f"  Average Balance Class-Clinic: {avg_balance_class_clinic_indiv_kisc} (count: {count_balance_class_clinic_indiv_kisc})")
    print(f"  Average Health Equity: {avg_health_equity_indiv_kisc} (count: {count_health_equity_indiv_kisc})")


# Display the averages and counts
display_averages_counts(
    avg_overall_satisf_all, avg_director_effective_all, avg_instruct_methods_all, avg_science_amount_all,
    avg_science_quality_all, avg_apply_learning_all, avg_balance_class_clinic_all, avg_health_equity_all,
    count_overall_satisf_all, count_director_effective_all, count_instruct_methods_all, count_science_amount_all,
    count_science_quality_all, count_apply_learning_all, count_balance_class_clinic_all, count_health_equity_all,
    avg_overall_satisf_indiv_kisc, avg_director_effective_indiv_kisc, avg_instruct_methods_indiv_kisc, avg_science_amount_indiv_kisc,
    avg_science_quality_indiv_kisc, avg_apply_learning_indiv_kisc, avg_balance_class_clinic_indiv_kisc, avg_health_equity_indiv_kisc,
    count_overall_satisf_indiv_kisc, count_director_effective_indiv_kisc, count_instruct_methods_indiv_kisc, count_science_amount_indiv_kisc,
    count_science_quality_indiv_kisc, count_apply_learning_indiv_kisc, count_balance_class_clinic_indiv_kisc, count_health_equity_indiv_kisc, kisc
)
